# **Tugas Praktikum 1**

---


**Kelompok**

**Nama Anggota 1**	:
Jeihan Shawmy Prasetya - 5025241132

<div>

**Nama Anggota 2**	:
Fathiya Nayla Husna Wibowo - 5025241204

## **Assignment**

Buatlah program computer yang dapat membaca inputan berupa program computer lain, dan dapat menghasilkan output berupa token-token (string-string yang terbaca) dan mengelompokkannya sesuai dengan sifat string tersebut:

1. Reserve words
2. Simbol dan tanda baca
3. Variabel
4. Kalimat matematika / ekspresi matematika

## 1. Import Library

In [ ]:
import re
import pandas as pd
from collections import defaultdict

try:
    import ipywidgets as widgets
    from IPython.display import display, HTML, clear_output
    WIDGETS_AVAILABLE = True
except Exception:
    WIDGETS_AVAILABLE = False
    print("ipywidgets tidak tersedia. Program tetap bisa dijalankan melalui fungsi analyze_code().")

## 2. Reserved Words, Operator, dan Simbol


In [ ]:
RESERVED_WORDS = {
    "Python": {
        "False", "None", "True", "and", "as", "assert", "async", "await", "break",
        "class", "continue", "def", "del", "elif", "else", "except", "finally",
        "for", "from", "global", "if", "import", "in", "is", "lambda", "nonlocal",
        "not", "or", "pass", "raise", "return", "try", "while", "with", "yield",
        "print", "input", "range", "len", "int", "float", "str", "list", "dict", "set"
    },
    "C/C++/Java": {
        "auto", "break", "case", "char", "class", "const", "continue", "default",
        "do", "double", "else", "enum", "extends", "final", "float", "for", "goto",
        "if", "implements", "import", "int", "interface", "long", "new", "private",
        "protected", "public", "return", "short", "signed", "static", "struct",
        "switch", "this", "throw", "throws", "try", "void", "volatile", "while",
        "System", "out", "println", "include", "using", "namespace", "std", "main"
    },
    "Pascal": {
        "program", "uses", "var", "begin", "end", "integer", "real", "boolean",
        "char", "string", "if", "then", "else", "while", "do", "for", "to",
        "downto", "repeat", "until", "procedure", "function", "array", "of",
        "record", "case", "const", "type", "writeln", "readln"
    }
}

MULTI_CHAR_SYMBOLS = [
    "==", "!=", "<=", ">=", "++", "--", "+=", "-=", "*=", "/=", "%=",
    "&&", "||", "::", "->", "=>", "**", "//", "<<", ">>", ":="
]

SINGLE_CHAR_SYMBOLS = set("+-*/%=<>(){}[];,.:!&|^~?@#")

MATH_OPERATORS = set(["+", "-", "*", "/", "%", "**", "//", "=", "==", "!=", "<", ">", "<=", ">=", "+=", "-=", "*=", "/=", "%=", ":="])

## 3. Membersihkan Komentar dan Memecah Token


In [ ]:
def remove_comments(code: str) -> str:
    """
    Menghapus komentar umum:
    - Python: # komentar
    - C/C++/Java: // komentar dan /* komentar */
    - Pascal: { komentar } dan (* komentar *)
    """
    # Hapus block comment C/Java
    code = re.sub(r"/\*.*?\*/", " ", code, flags=re.DOTALL)
    # Hapus block comment Pascal
    code = re.sub(r"\(\*.*?\*\)", " ", code, flags=re.DOTALL)
    code = re.sub(r"\{.*?\}", " ", code, flags=re.DOTALL)
    # Hapus line comment Python/C/Java
    code = re.sub(r"//.*", " ", code)
    code = re.sub(r"#.*", " ", code)
    return code


def tokenize(code: str):
    """
    Mengubah kode menjadi daftar token mentah.
    Token yang dikenali:
    - string literal
    - angka integer/desimal
    - identifier
    - operator/simbol multi-karakter
    - simbol satu karakter
    """
    code = remove_comments(code)

    token_pattern = r"""
        "(?:\\.|[^"\\])*"              |  # string double quote
        '(?:\\.|[^'\\])*'              |  # string single quote
        \b\d+\.\d+\b                   |  # bilangan desimal
        \b\d+\b                         |  # bilangan integer
        [A-Za-z_][A-Za-z0-9_]*          |  # identifier / reserved word
        ==|!=|<=|>=|\+\+|--|\+=|-=|\*=|/=|%=|&&|\|\||::|->|=>|\*\*|//|<<|>>|:= | # multi symbol
        [+\-*/%=<>(){}\[\];,.:!&|^~?@#]    # single symbol
    """

    tokens = []
    for match in re.finditer(token_pattern, code, re.VERBOSE):
        token = match.group(0).strip()
        if token:
            tokens.append({
                "token": token,
                "start": match.start(),
                "end": match.end()
            })
    return tokens

## 4. Klasifikasi Token

In [ ]:
def is_identifier(token: str) -> bool:
    return bool(re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", token))


def is_number(token: str) -> bool:
    return bool(re.fullmatch(r"\d+(\.\d+)?", token))


def is_string_literal(token: str) -> bool:
    return (
        len(token) >= 2 and
        ((token[0] == token[-1] == '"') or (token[0] == token[-1] == "'"))
    )


def classify_tokens(tokens, language="Python", custom_reserved_words=None):
    if custom_reserved_words is None:
        custom_reserved_words = set()

    reserved_words = set(RESERVED_WORDS.get(language, set())) | set(custom_reserved_words)

    rows = []
    grouped = defaultdict(list)

    for idx, item in enumerate(tokens, start=1):
        tok = item["token"]

        if tok in reserved_words:
            category = "Reserve Words"
        elif tok in MULTI_CHAR_SYMBOLS or tok in SINGLE_CHAR_SYMBOLS:
            category = "Simbol dan Tanda Baca"
        elif is_identifier(tok):
            category = "Variabel"
        elif is_number(tok):
            category = "Angka"
        elif is_string_literal(tok):
            category = "String Literal"
        else:
            category = "Tidak Dikenali"

        rows.append({
            "No": idx,
            "Token": tok,
            "Kategori": category,
            "Posisi Awal": item["start"],
            "Posisi Akhir": item["end"]
        })
        grouped[category].append(tok)

    return pd.DataFrame(rows), grouped


def detect_math_expressions(tokens):
    """
    Mendeteksi ekspresi matematika sederhana.
    Ide:
    - Kumpulkan rangkaian token yang terdiri dari identifier, angka, operator matematika, dan tanda kurung.
    - Jika rangkaian mengandung minimal satu operator matematika dan minimal dua operand, anggap sebagai ekspresi matematika.
    """
    expressions = []
    current = []

    allowed_parentheses = {"(", ")"}

    def flush_current():
        nonlocal current
        if len(current) >= 3:
            token_values = [x["token"] for x in current]
            has_operator = any(t in MATH_OPERATORS for t in token_values)
            operand_count = sum(1 for t in token_values if is_identifier(t) or is_number(t))
            if has_operator and operand_count >= 2:
                expr = " ".join(token_values)
                expressions.append(expr)
        current = []

    for item in tokens:
        tok = item["token"]
        if is_identifier(tok) or is_number(tok) or tok in MATH_OPERATORS or tok in allowed_parentheses:
            current.append(item)
        else:
            flush_current()

    flush_current()

    # Hilangkan duplikasi dengan mempertahankan urutan
    unique = []
    seen = set()
    for expr in expressions:
        if expr not in seen:
            seen.add(expr)
            unique.append(expr)
    return unique

## 5. Analisis


In [ ]:
def analyze_code(code: str, language="Python", custom_reserved_text=""):
    custom_reserved_words = {
        word.strip()
        for word in custom_reserved_text.replace("\n", ",").split(",")
        if word.strip()
    }

    tokens = tokenize(code)
    df_tokens, grouped = classify_tokens(tokens, language, custom_reserved_words)
    math_expressions = detect_math_expressions(tokens)

    result = {
        "tokens_table": df_tokens,
        "grouped_tokens": dict(grouped),
        "math_expressions": math_expressions,
        "summary": {
            "Jumlah Semua Token": len(df_tokens),
            "Jumlah Reserve Words": len(grouped.get("Reserve Words", [])),
            "Jumlah Simbol dan Tanda Baca": len(grouped.get("Simbol dan Tanda Baca", [])),
            "Jumlah Variabel": len(grouped.get("Variabel", [])),
            "Jumlah Ekspresi Matematika": len(math_expressions)
        }
    }
    return result


def show_analysis_result(result):
    print("RINGKASAN")
    for key, value in result["summary"].items():
        print(f"- {key}: {value}")

    print("\nKELOMPOK TOKEN")
    for category, values in result["grouped_tokens"].items():
        print(f"\n{category}:")
        print(", ".join(values) if values else "-")

    print("\nKALIMAT / EKSPRESI MATEMATIKA:")
    if result["math_expressions"]:
        for expr in result["math_expressions"]:
            print(f"- {expr}")
    else:
        print("- Tidak ditemukan")

    return result["tokens_table"]

## 6. Output



In [ ]:
if WIDGETS_AVAILABLE:
    contoh_program = """def hitung(a, b):
    hasil = a + b * 2
    if hasil >= 10:
        print(hasil)
    return hasil
"""

    input_program = widgets.Textarea(
        value=contoh_program,
        placeholder="Masukkan kode program di sini...",
        description="Kode:",
        layout=widgets.Layout(width="100%", height="220px")
    )

    pilihan_bahasa = widgets.Dropdown(
        options=list(RESERVED_WORDS.keys()),
        value="Python",
        description="Bahasa:"
    )

    tombol_analisis = widgets.Button(
        description="Analisis"
    )

    output = widgets.Output()

    def proses_analisis(button):
        with output:
            clear_output()

            kode = input_program.value
            bahasa = pilihan_bahasa.value

            if kode.strip() == "":
                print("Kode program masih kosong.")
                return

            hasil = analyze_code(kode, language=bahasa)

            print("Ringkasan")
            for nama, jumlah in hasil["summary"].items():
                print(f"{nama}: {jumlah}")

            print("\nEkspresi matematika")
            if hasil["math_expressions"]:
                for ekspresi in hasil["math_expressions"]:
                    print("-", ekspresi)
            else:
                print("- Tidak ditemukan")

            print("\nTabel token")
            display(hasil["tokens_table"])

    tombol_analisis.on_click(proses_analisis)

    display(widgets.VBox([
        pilihan_bahasa,
        input_program,
        tombol_analisis,
        output
    ]))

else:
    print("ipywidgets tidak tersedia. Jalankan: pip install ipywidgets pandas")

# **Tugas Praktikum 2**

---


**Kelompok**

**Nama Anggota 1**	:
Jeihan Shawmy Prasetya - 5025241132

<div>

**Nama Anggota 2**	:
Fathiya Nayla Husna Wibowo - 5025241204

## Assignment

Buatlah program komputer untuk mengotomasi FSM yang disediakan, yaitu sebuah mesin yang dapat menentukan apakah sebuah string merupakan anggota dari himpunan bahasa L = { x ∈ (0 + 1)+ | dengan karakter terakhir pada string x adalah 1 dan x tidak memiliki substring 00 }

## Definisi FSM

In [ ]:
S, A, B, C = "S", "A", "B", "C"

def fsm_check(string, show_trace=True):
    state = S
    trace = [state]

    for ch in string:
        if ch not in ['0', '1']:
            return False, ["Input tidak valid!"]

        if state == S:
            state = A if ch == '0' else B

        elif state == A:
            state = C if ch == '0' else B

        elif state == B:
            state = A if ch == '0' else B

        elif state == C:
            state = C

        trace.append(state)

    accepted = (state == B)
    return accepted, trace

## User Interface

In [ ]:
while True:
    print("\n=== FSM Checker ===")
    print("1. Cek string")
    print("2. Keluar")

    pilihan = input("Pilih: ")

    if pilihan == "1":
        s = input("Masukkan string (0/1): ")

        if len(s) == 0:
            print("String kosong tidak valid.")
            continue

        result, trace = fsm_check(s)

        print("Jejak state:", " -> ".join(trace))

        if result:
            print("String diterima")
        else:
            print("String ditolak")

    elif pilihan == "2":
        print("Program selesai")
        break

    else:
        print("Pilihan tidak valid!")


=== FSM Checker ===
1. Cek string
2. Keluar


In [ ]:
import ipywidgets as widgets
from IPython.display import display

text = widgets.Text(
    placeholder='Masukkan string...',
    description='Input:',
)

button = widgets.Button(description="Cek")

output = widgets.Output()

def on_click(b):
    with output:
        output.clear_output()
        res, trace = fsm_check(text.value)
        print("Trace:", " -> ".join(trace))
        print("Hasil:", "DITERIMA" if res else "DITOLAK")

button.on_click(on_click)

display(text, button, output)